# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. output: 1 1.3333333333333333 — None в Series превращается в NaN, isna() находит 1 пропуск; после fillna(0) получаем [1, 0, 3], среднее 4/3.
2. output: 2 — после strip и lower получаем ['a', 'a', 'b'], уникальных значений 2.
3. output: 3 — drop_duplicates() возвращает новый DataFrame, а результат никуда не сохранён, поэтому df остаётся с 3 строками.
4. output: 1005 — убираем пробел, '1 000' становится '1000', to_numeric превращает строки в числа: 1000 + 5 = 1005.
5. output: [9, 10] — to_datetime превращает строки в даты, .dt.month берёт номер месяца: 30 сентября → 9, 1 октября → 10.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [46]:
# 1
import pandas as pd

def clean(path):
    df = pd.read_csv(path)
    print(f"0. Исходно строк: {len(df)}")
    
    # 1. Суммы в числа (убираем пробелы, меняем запятые на точки, удаляем битые NaN)
    df["amount"] = (
        df["amount"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df = df.dropna(subset=["amount"])
    print(f"1. После очистки amount: {len(df)}")
    
    # 2. Города через словарь
    city_map = {
        "алматы": "Алматы",
        "almaty": "Алматы",
        "астана": "Астана",
        "astana": "Астана",
        "шымкент": "Шымкент",
        "shymkent": "Шымкент",
        "караганда": "Караганда",
        "атырау": "Атырау",
        "актобе": "Актобе",
    }
    df["city"] = df["city"].str.strip().str.lower().replace(city_map)
    df = df.dropna(subset=["city"])
    print(f"2. После нормализации city: {len(df)}")
    
    # 3. Дубликаты по order_id
    df = df.drop_duplicates(subset=["order_id"])
    print(f"3. После удаления дубликатов: {len(df)}")
    
    # 4. Приведение дат (смешанные форматы)
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
    df = df.dropna(subset=["order_date"])
    print(f"4. После очистки дат: {len(df)}")
    
    # 5. Удаление тестовых заказов TEST (отменённые заказы оставляем!)
    df = df[df["customer_id"].str.strip() != "TEST"]
    print(f"5. После удаления TEST: {len(df)}")
    
    # Обязательные проверки внутри функции
    assert df["amount"].isna().sum() == 0, "Ошибка: в amount остались NaN!"
    assert df["city"].isna().sum() == 0, "Ошибка: в city остались NaN!"
    assert df["order_id"].is_unique, "Ошибка: order_id не уникален!"
    
    return df

# Запуск и проверка итогового числа строк
df_clean = clean("orders.csv")
print(f"\nИтого строк в результате: {len(df_clean)}")

0. Исходно строк: 5463
1. После очистки amount: 5463
2. После нормализации city: 5463
3. После удаления дубликатов: 5283
4. После очистки дат: 5283
5. После удаления TEST: 5280

Итого строк в результате: 5280


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [ ]:
#2

paid_df = df_clean[df_clean["status"] == "оплачен"].copy()
paid_df["month"] = paid_df["order_date"].dt.month

rev_sep = paid_df.loc[paid_df["month"] == 9, "amount"].sum()
rev_oct = paid_df.loc[paid_df["month"] == 10, "amount"].sum()
total_change = (rev_oct - rev_sep) / rev_sep * 100

print(f"ВСЕГО | Сентябрь: {rev_sep:,.0f} | Октябрь: {rev_oct:,.0f} | Изменение: {total_change:+.1f}%")
print("-" * 75)

for city in sorted(paid_df["city"].unique()):
    city_df = paid_df[paid_df["city"] == city]
    
    c_sep = city_df.loc[city_df["month"] == 9, "amount"].sum()
    c_oct = city_df.loc[city_df["month"] == 10, "amount"].sum()
    c_change = (c_oct - c_sep) / c_sep * 100
    
    print(f"{city:<10} | Сентябрь: {c_sep:>10,.0f} | Октябрь: {c_oct:>10,.0f} | Изменение: {c_change:+.1f}%")

ВСЕГО | Сентябрь: 35,243,410 | Октябрь: 34,419,520 | Изменение: -2.3%
---------------------------------------------------------------------------
Актобе     | Сентябрь:  2,576,960 | Октябрь:  2,540,420 | Изменение: -1.4%
Алматы     | Сентябрь: 14,501,070 | Октябрь: 14,643,020 | Изменение: +1.0%
Астана     | Сентябрь:  9,034,220 | Октябрь:  8,850,530 | Изменение: -2.0%
Атырау     | Сентябрь:    923,990 | Октябрь:    969,860 | Изменение: +5.0%
Караганда  | Сентябрь:  3,648,520 | Октябрь:  3,371,680 | Изменение: -7.6%
Шымкент    | Сентябрь:  4,558,650 | Октябрь:  4,044,010 | Изменение: -11.3%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

## Журнал очистки для менеджера

| Шаг очистки | Сколько строк убрал | окт / сент 
| :--- | :---: | :---: |
| **1. Исходные данные** (как считал менеджер) | 0 (осталось 5 463) | **−22,7%** 
| **2. Удаление дубликатов**| **180** (осталось 5 283) | **−19,6%** 
| **3. Удаление TEST**| **3** (осталось 5 280) | **−2,3%** 

**Вывод для менеджера:** сильнее всего цифру исказили тестовые заказы (`TEST`) — три искусственных заказа по 9,99 млн ₸ создали большую часть ложного падения (при удалении `TEST` в первую очередь показатель сразу меняется с −22,7% до −8,2%, а после удаления дублей снижается до реальных −2,3%).


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [ ]:
# 4

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [ ]:
# 5

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8